# 08 — Failure Modes

**Purpose:** Discover and catalogue systematic failure modes that reduce measurement confidence.

**Output:** `outputs/failure_modes.parquet`

**Granularity:** Record level — primary key: `record_id`

**Data Contract:** `DATA_CONTRACT.md` §15

**Allowed failure categories:** `bw`, `hfn`, `pli`, `em`, `arrhythmia`,
`twave_ambiguity`, `lead_disagreement`, `delineation_failure`


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()

ALLOWED_FAILURE_TYPES = [
    "bw","hfn","pli","em","arrhythmia","twave_ambiguity","lead_disagreement","delineation_failure"
]

inventory = pd.read_csv("../outputs/inventory.csv")
sq_df     = pd.read_parquet("../outputs/signal_quality_features.parquet")
twave_df  = pd.read_parquet("../outputs/twave_features.parquet")
delin_df  = pd.read_parquet("../outputs/delineation_features.parquet")
clinical  = pd.read_parquet("../outputs/clinical_context.parquet")

print(f"Records: {len(inventory)}")


/tmp/ipykernel_2915/2232101902.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


Records: 70


## Failure Mode Detection Rules

Each rule maps to a `failure_type` from the allowed vocabulary.
Scores are in [0, 1] where 1 = severe failure.


In [3]:
# Aggregate signal quality to record level
sq_rec = sq_df.groupby("record_id").agg(
    mean_bw_index=("bw_index","mean"),
    mean_hfn_index=("hfn_index","mean"),
    mean_pli_index=("pli_index","mean"),
    mean_em_index=("electrode_motion_index","mean"),
    min_sqs=("signal_quality_score","min"),
).reset_index()

# Aggregate delineation to record level
delin_rec = delin_df.groupby("record_id").agg(
    mean_bc=("boundary_confidence","mean"),
    max_t_unc=("t_end_uncertainty_ms","max"),
).reset_index()

# Aggregate T-wave to record level
tw_rec = twave_df.groupby("record_id").agg(
    mean_ambiguity=("t_end_ambiguity_score","mean"),
    max_ambiguity=("t_end_ambiguity_score","max"),
    biphasic_rate=("biphasic_flag","mean"),
).reset_index()

# Merge
merged = (inventory[["record_id","dataset_name"]]
          .merge(sq_rec,   on="record_id", how="left")
          .merge(delin_rec,on="record_id", how="left")
          .merge(tw_rec,   on="record_id", how="left")
          .merge(clinical[["record_id","arrhythmia_flag"]], on="record_id", how="left"))

print(f"Merged shape: {merged.shape}")


Merged shape: (70, 13)


In [4]:
THRESHOLDS = {
    "bw":               ("mean_bw_index",    0.15),
    "hfn":              ("mean_hfn_index",   0.20),
    "pli":              ("mean_pli_index",   0.10),
    "em":               ("mean_em_index",    0.20),
    "delineation_failure": ("max_t_unc",     15.0),
    "twave_ambiguity":  ("mean_ambiguity",   0.50),
}

rows = []
for _, rec in merged.iterrows():
    failures = []

    for ftype, (col, thresh) in THRESHOLDS.items():
        val = rec.get(col, np.nan)
        if pd.isna(val):
            continue
        if val >= thresh:
            score = float(np.clip((val - thresh) / (thresh + 1e-9), 0, 1))
            failures.append((ftype, val, score))

    # Arrhythmia
    if rec.get("arrhythmia_flag", False):
        failures.append(("arrhythmia", 1.0, 0.6))

    # Lead disagreement proxy
    sq_min = rec.get("min_sqs", 1.0)
    if pd.notna(sq_min) and sq_min < 0.5:
        score = float(np.clip(1.0 - sq_min, 0, 1))
        failures.append(("lead_disagreement", 1.0 - sq_min, score))

    if not failures:
        failures.append(("bw", 0.0, 0.0))  # No failure

    failures.sort(key=lambda x: x[2], reverse=True)
    primary_type, primary_val, primary_score = failures[0]
    conf_impact = float(np.clip(primary_score * 0.7, 0, 1))

    rows.append({
        "record_id":       rec["record_id"],
        "failure_type":    primary_type,
        "failure_score":   primary_score,
        "confidence_impact": conf_impact,
        "root_cause_rank": len(failures),
        "pipeline_version": PIPELINE_VERSION,
        "processing_timestamp": TIMESTAMP,
    })

fm_df = pd.DataFrame(rows)
print(f"Shape: {fm_df.shape}")
print(fm_df["failure_type"].value_counts().to_string())


Shape: (70, 7)
failure_type
delineation_failure    66
bw                      4


## Schema Validation

In [5]:
REQUIRED_FM_COLS = ["record_id","failure_type","failure_score","confidence_impact","root_cause_rank"]
missing = [c for c in REQUIRED_FM_COLS if c not in fm_df.columns]
assert not missing, f"Missing: {missing}"

invalid_types = set(fm_df["failure_type"]) - set(ALLOWED_FAILURE_TYPES)
assert not invalid_types, f"Invalid failure types: {invalid_types}"
assert fm_df["failure_score"].between(0,1).all(), "failure_score out of [0,1]"
assert fm_df["confidence_impact"].between(0,1).all(), "confidence_impact out of [0,1]"

print("✓ Schema validation passed")
print(f"  All failure types valid: {sorted(fm_df.failure_type.unique())}")


✓ Schema validation passed
  All failure types valid: ['bw', 'delineation_failure']


## Failure Mode Distribution

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fm_df["failure_type"].value_counts().plot.bar(ax=axes[0], color="#d62728", edgecolor="white")
axes[0].set_title("Failure Mode Distribution")
axes[0].tick_params(axis='x', rotation=30)
fm_df["confidence_impact"].hist(bins=20, ax=axes[1], color="#ff7f0e", edgecolor="white")
axes[1].set_title("Confidence Impact")
axes[1].set_xlabel("Impact Score (0-1)")
plt.tight_layout()
plt.savefig("../outputs/failure_modes_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Figure saved.


/tmp/ipykernel_2915/122707598.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [7]:
fm_df.to_parquet("../outputs/failure_modes.parquet", index=False)
print("✓ failure_modes.parquet →", fm_df.shape)


✓ failure_modes.parquet → (70, 7)
